# DPO v2: SFT v3 LoRA (Qwen2.5-1.5B) ustida

v1 dan farqi: har promptga 8 ta javob; rad etishda chosen — doim toza qat'iy jumla; chosen va rejected uzunligi yaqin (uzunlik bo'yicha o'rganmasin); loss'ga chosen uchun SFT qismi qo'shilgan; learning rate 5e-6.

1. SFT v3 modeldan har bir promptga 8 ta javob olinadi (776 prompt).
2. Qoidalar javoblarni baholab, chosen/rejected juftliklarini tuzadi.
3. DPO SFT v3 adapterni shu juftliklarda o'qitishni davom ettiradi.
4. 68 ta test savolida baholanadi.

**Boshlashdan oldin:**

- **Settings -> Accelerator -> GPU T4 x2**, **Internet -> On**
- **SFT v3 adapter:** o'ng tomonda **Add Input -> Upload -> New Dataset**, kompyuteringizdagi `sft_v3/sft_v3_adapters.zip` ni yuklang (v1 da yaratilgan `sft-v3-adapters` datasetini qayta qo'shsangiz ham bo'ladi). Notebook uni o'zi topadi. Yuklamasangiz, SFT v3 LoRA shu yerda qaytadan o'qitiladi (taxminan 40 daqiqa qo'shiladi).
- **Save Version -> Save & Run All (Commit)** bilan ishga tushiring: natija **Output** bo'limida doimiy saqlanadi.

## 1. GPU borligini tekshirish

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Loyiha va base modelni yuklab olish

In [ ]:
%cd /kaggle/working
!rm -rf Post_Training
!GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/davronbek-malikov/Post_Training.git
%cd Post_Training
!pip -q install -U transformers peft bitsandbytes accelerate
!pip -q uninstall -y torchao  # Kaggle'dagi eski torchao peft bilan adapter yuklashni buzadi

from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen2.5-1.5B", local_dir="Qwen2.5-1.5B")
!ls -lh Qwen2.5-1.5B/*.safetensors
!wc -l dpo/data/prompts.jsonl

## 3. SFT v3 LoRA adapter

Avval yuklangan dataset ichidan qidiriladi; topilmasa, qaytadan o'qitiladi.

In [ ]:
import glob, os, shutil, subprocess, zipfile
target = "chat_ui/adapters/sft-lora-v3"
found = glob.glob("/kaggle/input/**/sft-lora-v3/adapter_config.json", recursive=True)
if not found:   # zip ochilmagan holda yuklangan bo'lsa
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        with zipfile.ZipFile(z) as f:
            names = [n for n in f.namelist() if "sft-lora-v3/" in n]
            if names:
                f.extractall("/kaggle/working/unzipped", members=names)
    found = glob.glob("/kaggle/working/unzipped/**/sft-lora-v3/adapter_config.json", recursive=True)
if found:
    shutil.copytree(os.path.dirname(found[0]), target, dirs_exist_ok=True)
    print("SFT v3 adapter topildi:", os.path.dirname(found[0]))
else:
    print("Adapter topilmadi: SFT v3 LoRA qaytadan o'qitiladi...")
    subprocess.run(["python", "sft_v2/train.py", "--method", "lora", "--data", "sft_v3/data",
                    "--name", "sft-lora-v3", "--epochs", "3", "--rank", "64"], check=True)
print(os.listdir(target))

## 4. SFT v3 modeldan javoblar olish

Har bir promptga 8 ta javob, sampling bilan (har safar boshqacha).

In [ ]:
!python dpo/generate.py

## 5. Chosen / rejected juftliklari

Jadvalda: har bir vazifadan nechta juftlik chiqdi, ulardan nechtasida chosen modelning o'z javobi (`model`) yoki SFT datadagi javob (`ref`). Pastdagi misollarni o'qib chiqing: noto'g'ri belgilangan juftlik DPO'ga zarar qiladi.

In [ ]:
!python dpo/label.py
!rm -f /kaggle/working/dpo_v2_results.zip
!zip -q /kaggle/working/dpo_v2_results.zip dpo/data_v2/generations.jsonl dpo/data_v2/train.jsonl dpo/data_v2/val.jsonl

## 6. DPO bilan o'qitish

Boshida `val loss 0.693` bo'lishi kerak (policy va reference hali bir xil). Keyin:
- `chosen` reward juda manfiy bo'lib ketmasligi kerak (v1 da -0.7 gacha tushgan); `sft` ko'tarilmasligi kerak;
- `acc` 0.5 dan yuqoriga ko'tarilishi kerak (model chosen'ni rejected'dan afzal ko'ryapti);
- `margin` asta o'sishi kerak; juda tez katta bo'lib ketsa — overfitting.

In [ ]:
!python dpo/train_dpo.py

In [ ]:
!zip -q -r /kaggle/working/dpo_v2_results.zip chat_ui/adapters/sft-lora-dpo-v2
!ls -lh /kaggle/working/dpo_v2_results.zip

## 7. Baholash: 68 ta test savoli

SFT v3 LoRA: 41/68. DPO v1: 40/68.

In [ ]:
!python eval/run_eval.py --base Qwen2.5-1.5B --model sft-lora-dpo-v2 | tail -12
!zip -q /kaggle/working/dpo_v2_results.zip eval/results/sft-lora-dpo-v2.json

## 8. Yuklab olish

Notebook tugagach: notebook sahifasi -> **Output** -> `dpo_v2_results.zip` -> **Download**. Uni loyihaning `dpo/` papkasiga qo'ying.